# NAMAA Kaggle Test V2
T4 GPU only. Fix torch/torchaudio ABI mismatch first. WAV output under /kaggle/working.


In [ ]:
import os,sys,subprocess,time
from pathlib import Path
print("PREPARE_TTS",flush=True)
subprocess.check_call([sys.executable,"-m","pip","install","-q","chatterbox-tts","huggingface_hub","safetensors","soundfile"])
import torch
print("PYTHON",sys.version, "TORCH",torch.__version__,"CUDA",torch.cuda.is_available(),flush=True)
# Align torchaudio ABI with the torch wheel pip actually installed:
v=torch.__version__.split("+")[0]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--force-reinstall","torchaudio=="+v])
import torchaudio
print("TORCHAUDIO",torchaudio.__version__,flush=True)
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from chatterbox import mtl_tts
if not torch.cuda.is_available():raise RuntimeError("Kaggle GPU not enabled")
print("GPU",torch.cuda.get_device_name(0),flush=True)
t=time.monotonic()
folder=snapshot_download(repo_id="NAMAA-Space/NAMAA-Egyptian-TTS",repo_type="model")
print("MODEL_FILES",os.listdir(folder),flush=True)
model=mtl_tts.ChatterboxMultilingualTTS.from_pretrained(device="cuda")
state=load_file(os.path.join(folder,"t3_mtl23ls_v2.safetensors"),device="cpu")
model.t3.load_state_dict(state)
model.t3.to("cuda").eval()
with torch.inference_mode():
  wav=model.generate("إزيك يا جماعة، بنجرّب الصوت المصري، وعايزين نشوف جودة النطق.",language_id="ar")
out="/kaggle/working/namaa_test.wav"
torchaudio.save(out,wav.detach().cpu(),model.sr)
print("SUCCESS",out,"BYTES",Path(out).stat().st_size,"AUDIO_SECONDS",round(wav.shape[-1]/model.sr,2),"TOTAL_SECONDS",round(time.monotonic()-t,1),flush=True)